In [81]:
import pandas as pd
#### Data loader
Column_name = ['supermarket',"prices_(£)",'names','date','category','own_brand']
sains_data = pd.read_csv('/Users/omkar/Documents/SmartShop/data/All_Data_Sains.csv',usecols=Column_name) 


In [92]:
def data_subset(df:pd.DataFrame)->pd.DataFrame:

    ### Filter products with own brand = False.
    if "own_brand" in df.columns:
        df = pd.DataFrame(df[df['own_brand'] == False]['names'].unique(),columns=['names'])
    return df

In [98]:
products_df = data_subset(sains_data)

In [77]:
#### config.py

# config.py

MODEL = "llama-3.1-8b-instant"
BATCH_SIZE = 50
SLEEP_TIME = 0.2

TEXT_COLUMN = "names"
OUTPUT_FILE = "output.csv"

In [94]:
# api.py

from groq import Groq
# from config import MODEL

client = Groq(api_key="YOUR_GROQ_API_KEY")

def call_groq(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,
        max_completion_tokens=750,
        stream=False
    )
    
    return response.choices[0].message.content

In [80]:
# parser.py

import json
import re

def safe_parse(content):
    try:
        match = re.search(r"\{[\s\S]*\}", content)
        if match:
            return json.loads(match.group())
    except:
        pass
    return {}

In [100]:
# processor.py

import json
# from api import call_groq
# from parser import safe_parse

def process_batch(batch_df, text_col):
    
    items = {
        str(idx): name
        for idx, name in zip(batch_df.index, batch_df[text_col])
    }

    prompt = f"""
Extract brand names.

Return ONLY valid JSON:
id -> brand

Rules:
- No explanation
- No markdown
- If unknown: "Unknown"

Products:
{json.dumps(items)}
"""

    content = call_groq(prompt)
    return safe_parse(content)

In [101]:
# pipeline.py

# import time
# import pandas as pd
# # from config import BATCH_SIZE, SLEEP_TIME
# # from processor import process_batch

# def run_pipeline(df, text_col):
    
#     df = df.copy()

#     if "Brand" not in df.columns:
#         df["Brand"] = None

#     total = len(df)

#     for i in range(0, total, BATCH_SIZE):
        
#         batch = df.iloc[i:i+BATCH_SIZE]
#         result = process_batch(batch, text_col)

#         for idx, brand in result.items():
#             try:
#                 df.at[int(idx), "Brand"] = brand
#             except:
#                 pass

#         print(f"Processed {i + len(batch)} / {total}")

#         time.sleep(SLEEP_TIME)

#     return df



# pipeline.py (PARALLEL VERSION)

import time
from concurrent.futures import ThreadPoolExecutor, as_completed
# from config import BATCH_SIZE, SLEEP_TIME
# from processor import process_batch

def run_pipeline(df, text_col, max_workers=2):
    
    df = df.copy()

    if "Brand" not in df.columns:
        df["Brand"] = None

    batches = [
        df.iloc[i:i+BATCH_SIZE]
        for i in range(0, len(df), BATCH_SIZE)
    ]

    print(f"Total batches: {len(batches)} | Workers: {max_workers}")

    def worker(batch):
        return process_batch(batch, text_col)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(worker, batch): batch for batch in batches}

        for i, future in enumerate(as_completed(futures)):
            result = future.result()

            for idx, brand in result.items():
                try:
                    df.at[int(idx), "Brand"] = brand
                except:
                    pass

            print(f"Completed batch {i+1}/{len(batches)}")

            time.sleep(SLEEP_TIME)

    return df

In [ ]:
# main.py

import pandas as pd
# from pipeline import run_pipeline
# from config import TEXT_COLUMN, OUTPUT_FILE

if __name__ == "__main__":
    
    print("Loading data...")
    df = pd.read_csv("input.csv")   # change this

    print("Running pipeline...")
    df = run_pipeline(df, TEXT_COLUMN)

    print("Saving output...")
    df.to_csv(OUTPUT_FILE, index=False)

    print("Done ✅")

In [103]:
run_pipeline(df=products_df, text_col='names',max_workers=2)

Total batches: 519 | Workers: 2
Completed batch 1/519
Completed batch 2/519


KeyboardInterrupt: 